In [ ]:
import gdown
import pandas as pd
import requests
from bs4 import BeautifulSoup

def get_page_cnt(isbn):

    url = 'http://www.yes24.com/Product/Search?domain=BOOK&query={}'

    # format() 이용하여 isbn값을 url 변수에 전달
    r = requests.get(url.format(isbn))

    # html parsing에 사용할 parser
    soup = BeautifulSoup(r.text, 'html.parser')


    # 쪽 수 첫 번째 태그 위치찾기
    prd_info = soup.find('a', attrs={'class' : 'gd_name'})
    if prd_info == None:
        return ''

    # 쪽 수 상세 페이지 주소
    url = 'http://www.yes24.com' + prd_info['href']
    r = requests.get(url)
    soup = BeautifulSoup(r.text, 'html.parser')


    #쪽 수 / 품목정보를 포함하고 있는 지정이름 태그 모두 찾기
    prd_detail = soup.find('div', attrs={'id' : 'infoset_specific'})
    # 테이블에 있는 tr태그 가져오기
    prd_tr_list = prd_detail.find_all('tr')

    # 쪽수가 있는 th를 순회하며 td에서 필요한 값 반환
    for tr in prd_tr_list:
        if tr.find('th').get_text() == '쪽수, 무게, 크기':
            return tr.find('td').get_text().split()[0]
        
    return ''

def get_page_cnt2(row):
    isbn = row['isbn13']
    return get_page_cnt(isbn)

gdown.download('https://bit.ly/3q9SZix', '20s_best_book.json', quiet=False)
#  ㄴ검색 결과 페이지 html 가져오기

# json 파일로 읽어오기
books_df = pd.read_json('20s_best_book.json')

# books_df 의 'no' 열부터 'isbn13'열 까지만 선택
books = books_df.loc[:, 'no' : 'isbn13']

# 10개의 책 데이터프레임 가져오기
top10_books = books.head(10)

# 각 행에 함수 적용해서 10개의 책 쪽수 자동화
# axis = 1 : 행 / axis = 0 : 열
page_count = top10_books.apply(get_page_cnt2, axis=1)
#  ㄴ vv동일코드vv
# page_count = top10_books.apply(lambda row: get_page_cnt(row['isbn13']), axis=1)


page_count.name = 'page_count'
# print(page_count)

# 데이터프레임 시리즈 합치기 / 인덱스를 기준으로 합치기
top10_with_page_count = pd.merge(top10_books, page_count, left_index=True, right_index=True)
top10_with_page_count

Downloading...
From: https://bit.ly/3q9SZix
To: c:\Users\pili4\python-data-analysis-study\Ch02_data_collection\20s_best_book.json
100%|██████████| 92.9k/92.9k [00:00<00:00, 562kB/s]


,no,ranking,bookname,authors,publisher,publication_year,isbn13,page_count
0,1,1,우리가 빛의 속도로 갈 수 없다면 :김초엽 소설,지은이: 김초엽,허블,2019,9791190090018,344쪽
1,2,2,달러구트 꿈 백화점.이미예 장편소설,지은이: 이미예,팩토리나인,2020,9791165341909,300쪽
2,3,3,지구에서 한아뿐 :정세랑 장편소설,지은이: 정세랑,난다,2019,9791188862290,228쪽
3,4,4,"시선으로부터, :정세랑 장편소설",지은이: 정세랑,문학동네,2020,9788954672214,340쪽
4,5,5,아몬드 :손원평 장편소설,지은이: 손원평,창비,2017,9788936434267,264쪽
5,6,6,피프티 피플 :정세랑 장편소설,지은이: 정세랑,창비,2016,9788936434243,396쪽
6,7,7,목소리를 드릴게요 :정세랑 소설집,지은이: 정세랑,아작,2020,9791165300005,272쪽
7,8,8,나미야 잡화점의 기적 :히가시노 게이고 장편소설,지은이: 히가시노 게이고 ;옮긴이: 양윤옥,현대문학,2012,9788972756194,
8,9,9,선량한 차별주의자,김지혜 지음,창비,2019,9788936477196,244쪽
9,10,9,쇼코의 미소 :최은영 소설,지은이: 최은영,문학동네,2016,9788954641630,296쪽
